### Sequence-to-Sequence (Seq2Seq) models and their architecture
* WHatare Seq2Seq models?
    * Map an input sequence to an output sequence of different lengths
    * Widley used for tasks like language translation, text summarization , speech-to-text , and chatbots
    * Architecture
        * Encoder:
            * Process the input sequence and encodes it into a fixed length vector (Context Vector)
        * Decoder
            * Takes the context vector as input and generatews the output sequence , step by step

### Encoder Decoder framework for Seq2Seq Tasks
* How it works
    * Encoder
        * Sequentially Process the input sequence using RNN, LSTM or GRU
        * Produces a context vector representing the entire input sequence
    * Decoder
        * Initializes its hidden state with the encoder's cotnext vector
        * generates the output sequence one token at a time
        * Predicts the next token using the previously generated tokens

### Attention Mechanism Overview
* WHy attention ? 
    * standart Seq2Seq models compress the entire inmput sequence into a fixed lenght vector , whcih can lead to information loss for long sequences
    * Attention Mechanism dynamically focuses on different parts of the input sequence when generating each output token
* How does it works
    * Calculates a weight (or score) for each input token based on its relevance to the current decoder state
    * Outputs a weighted sum of the encoder outputs, creating a context vector for each decoder step
    


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset

import numpy as np

In [2]:
english_sentences = ["hello", "how are you", "good morning", "thank you", "good night"]
french_sentences = ["bonjour", "comment ça va", "bon matin", "merci", "bonne nuit"]

In [4]:
# vocabulary  and tokenization
def build_vocab(sentences):
    vocab = {"<PAD>":0, "<SOS>":1, "<EOS>":2, "<UNK>":3}

    for sentence in sentences:
        for word in sentence.split():
            if word not in vocab:
                vocab[word]= len(vocab)
    return vocab

english_vocab = build_vocab(english_sentences)
french_vocab = build_vocab(french_sentences)

# Tokenize and pad sentence
def tokenize(sentences, vocab, max_len):
    tokenized = []
    for sentence in sentences:
        tokens = [vocab.get(word, vocab["<UNK>"]) for word in sentence.split()]
        tokens = [vocab["<SOS>"]]+ tokens + [vocab["<EOS>"]]
        tokens += [vocab["<PAD>"]] * (max_len - len(tokens))
        tokenized.append(tokens)
    return np.array(tokenized)

max_len_eng = max(len(sentence.split()) for sentence in english_sentences) + 2

max_len_fr = max(len(sentence.split()) for sentence in french_sentences) + 2

english_data = tokenize(english_sentences,english_vocab, max_len=max_len_eng)
french_data = tokenize(french_sentences,french_vocab,max_len=max_len_fr)

In [31]:
class TranslationDataset(Dataset):
    def __init__(self, source_data, target_data):
        self.src_data = source_data
        self.tar_data = target_data
    
    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, key):
        return torch.tensor(self.src_data[key]), torch.tensor(self.tar_data[key])
    
dataset = TranslationDataset(english_data, french_data)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

class Encoder(nn.Module):
    def __init__(self, input_dim, embed_dim, hidden_dim,num_layers):
        super(Encoder,self).__init__()
        self.embedding = nn.Embedding(input_dim,embedding_dim=embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)

    def forward(self,x):
        embedded = self.embedding(x)
        _, (hidden,cell)= self.lstm(embedded)
        return hidden,cell
    
class Decoder(nn.Module):
    def __init__(self,output_dim, embedding_dim, hidden_dim, num_layers):
        super(Decoder,self).__init__()
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self,   x,hidden, cell):
       
        x = x.unsqueeze(1)
        embedded = self.embedding(x)
        outputs, (hidden,cell) = self.lstm(embedded, (hidden, cell))
        predictions = self.fc(outputs.squeeze(1))
        return predictions, hidden,cell
    
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        tgt_vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)

        hidden,cell = self.encoder(src)
        input = tgt[:, 0]
        for t in range(1, tgt_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output

            top1 = output.argmax(1)
            input = tgt[:, t] if torch.rand(1).item()< teacher_forcing_ratio else top1
        return outputs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = len(english_vocab)
output_dim = len(french_vocab)
embed_dim = 64
hidden_dim = 128
num_layers = 2

encoder = Encoder(input_dim, embed_dim,hidden_dim,num_layers)
decoder = Decoder(output_dim, embed_dim,hidden_dim, num_layers)

model = Seq2Seq(encoder, decoder,device)

In [32]:
optimizer = optim.Adam(model.parameters(), lr = 0.001)
criterion = nn.CrossEntropyLoss(ignore_index=french_vocab["<PAD>"])

def train(model, dataloader, optimizer, criterion, device, num_epochs=20):
    model.train()
    for epoch in range(num_epochs):
        epoch_loss= 0
        for src, tgt in dataloader:
            src, tgt = src.to(device), tgt.to(device)

            optimizer.zero_grad()
            output = model(src, tgt)

            output = output[: ,1: ].reshape(-1, output.shape[2])
            tgt = tgt[:, 1:].reshape(-1)
            
            loss = criterion(output,tgt)

            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        print(f"Epoch {epoch+1}: Loss: {epoch_loss/len(dataloader)}")
train(model, dataloader,optimizer,criterion,device)


Epoch 1: Loss: 2.587059736251831
Epoch 2: Loss: 2.5349082152048745
Epoch 3: Loss: 2.4724125067392984
Epoch 4: Loss: 2.33921750386556
Epoch 5: Loss: 2.142197291056315
Epoch 6: Loss: 1.8838293552398682
Epoch 7: Loss: 1.8789589405059814
Epoch 8: Loss: 1.8626829385757446
Epoch 9: Loss: 1.6583385864893596
Epoch 10: Loss: 1.5008855263392131
Epoch 11: Loss: 1.4538308382034302
Epoch 12: Loss: 1.295256535212199
Epoch 13: Loss: 1.1446893612543743
Epoch 14: Loss: 1.0068147381146748
Epoch 15: Loss: 0.7696331342061361
Epoch 16: Loss: 0.7792234023412069
Epoch 17: Loss: 0.6937991778055826
Epoch 18: Loss: 0.6586248079935709
Epoch 19: Loss: 0.5876567661762238
Epoch 20: Loss: 0.4883378843466441


In [33]:
def translate_sentence(model, sentence, english_vocab, french_vocab, max_len_eng, max_len_fr):
    model.eval()
    tokens = [english_vocab.get(word, english_vocab["<UNK>"]) for word in sentence.split()]
    tokens = [english_vocab["<SOS>"]] +tokens + [english_vocab["<EOS>"]]
    src = torch.tensor(tokens).unsqueeze(0).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(src)
        tgt_vocab ={v:k for k,v in french_vocab.items()}
        tgt_indices = [french_vocab["<SOS>"]]
        for _ in range(max_len_fr):
            tgt_tensor = torch.tensor([tgt_indices[-1]]).to(device)
            output, hidden, cell = model.decoder(tgt_tensor, hidden, cell)
            pred = output.argmax(1).item()
            tgt_indices.append(pred)
            if pred == french_vocab["<EOS>"]:
                break
        
    translated_sentence = [tgt_vocab[i] for i in tgt_indices[1: -1]]
    print(translated_sentence)
    return " ".join(translated_sentence)

#  Test Translation

sentence= "thank you"
translation = translate_sentence(model, sentence, english_vocab, french_vocab, max_len_eng, max_len_fr)
print(translation)

['merci']
merci
